<a href="https://colab.research.google.com/github/Mitochondrion3000/epic-competiton/blob/claude/gracious-brown-4ybsmj/simple_model/step6_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Простая интерпретируемая модель. Шаг 6: финальная версия и проверка пайплайна

Всё, что помогло на шагах 3–5, плюс три дешёвых улучшения из эксперимента после шага 5:
* признаки: 61 понятный признак + логотип (буква на −50..+30) + **пары букв** (динуклеотид на −5..+30);
* **все строки** таблицы без прореживания (больше данных помогало сильнее всего);
* для теста модель учится на **всех train-контигах** (`fit` + `val`), и **три модели с разными сидами** усредняются.

Честная оценка остаётся: сначала одна модель учится только на `fit`, по ней считаем плотную валидацию и выбираем
квантование сабмита. Потом те же настройки, но на всех train-контигах и с тремя сидами, идут в тест.

В конце **проверка пайплайна по выданным организаторами файлам**: официальный `validate` для примера сабмита
`template.test.txt.gz` и для нашего файла; файл для train-шаблона (`template.train.bed.gz`) против ответов
`csRNA-train.txt.gz`; официальный `scorer.py` против нашего `scoring.py`.

## 0. Настройки и данные

In [ ]:
import os, sys, gzip, json, hashlib, tarfile, time, subprocess, urllib.request
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
try:
    import lightgbm as lgb
except ImportError:
    os.system("pip install -q lightgbm"); import lightgbm as lgb

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

SPECIES = os.environ.get("EPIC_SPECIES", "jaNemVect1.1")
ON_COLAB = "google.colab" in sys.modules
if ON_COLAB:
    from google.colab import drive; drive.mount("/content/drive")
DATA_DIR = os.environ.get("EPIC_DATA_DIR", "/content/epic")
OUT_DIR = os.environ.get("EPIC_OUT_DIR", "/content/drive/MyDrive/epic_simple" if ON_COLAB else "/content/epic_simple")
BASE, FP = f"{DATA_DIR}/{SPECIES}", f"{DATA_DIR}/{SPECIES}/for_participants"
N_SEEDS = int(os.environ.get("EPIC_SEEDS", 3))                         # сколько моделей усредняем для теста
MAX_TRAIN_ROWS = int(os.environ.get("EPIC_MAX_TRAIN_ROWS", 9_000_000))  # ограничение по памяти для больших геномов

RAW = "https://raw.githubusercontent.com/Mitochondrion3000/epic-competiton/claude/gracious-brown-4ybsmj/simple_model/"
for mod in ("features.py", "motifs.py", "scoring.py"):
    if not os.path.exists(mod): urllib.request.urlretrieve(RAW + mod, mod)
import features as F, motifs as M, scoring as S

table = pd.read_parquet(f"{OUT_DIR}/{SPECIES}.step1_table.parquet")
meta = json.load(open(f"{OUT_DIR}/{SPECIES}.step1_meta.json"))
P = json.load(open(f"{OUT_DIR}/{SPECIES}.step3b_params.json"))
BASE_FEATURES = P["features"]
PARAMS = {"pwm": np.array(P["pwm"]), "kmers": [tuple(k) for k in P["kmers"]], "tf": P["tf"], "wide": P["wide"], "local": P["local"]}
EXTRA = F.logo_names() + F.dinuc_names()
ALL = BASE_FEATURES + EXTRA
print(f"таблица {len(table):,} строк; признаков: {len(BASE_FEATURES)} + логотип {len(F.logo_names())} + пары букв {len(F.dinuc_names())} = {len(ALL)}")
print(f"моделей для теста: {N_SEEDS}")

In [ ]:
def download_dataset(species, data_dir):
    if os.path.isdir(f"{data_dir}/{species}"): return
    import requests
    os.makedirs(data_dir, exist_ok=True)
    rec = requests.get("https://zenodo.org/api/records/22285753", timeout=60).json()
    f = [x for x in rec["files"] if x["key"] == f"{species}.tgz"][0]
    dst, md5 = f"{data_dir}/{species}.tgz", hashlib.md5()
    with requests.get(f["links"]["self"], stream=True, timeout=120) as r, open(dst, "wb") as fh:
        r.raise_for_status()
        for chunk in tqdm(r.iter_content(1 << 22), total=f["size"] // (1 << 22) + 1, desc=f"скачиваю {species}"):
            fh.write(chunk); md5.update(chunk)
    assert md5.hexdigest() == f.get("checksum", "md5:").split(":")[-1], "MD5 не совпал"
    with tarfile.open(dst) as tf:
        try: tf.extractall(data_dir, filter="data")
        except TypeError: tf.extractall(data_dir)
    os.remove(dst)

CODE = np.full(256, 4, np.uint8)
for i, b in enumerate("ACGT"):
    CODE[ord(b)] = CODE[ord(b.lower())] = i

def read_genome(path, keep):
    seq, low, name, chunks = {}, {}, None, []
    def flush():
        if name in keep:
            raw = np.frombuffer(b"".join(chunks), np.uint8)
            seq[name], low[name] = CODE[raw], raw >= ord("a")
    with gzip.open(path, "rb") as fh:
        for line in fh:
            if line.startswith(b">"):
                flush(); name, chunks = line[1:].split()[0].decode(), []
            elif name in keep:
                chunks.append(line.rstrip())
    flush()
    return seq, low

def read_template(path):
    t = pd.read_csv(path, sep="\t", header=None, usecols=[0, 1, 2, 5], names=["chrom", "start", "end", "strand"], dtype={"chrom": str, "strand": str})
    t["len"] = t.end - t.start; t["off"] = np.r_[0, np.cumsum(t.len.values)[:-1]]
    return t

download_dataset(SPECIES, DATA_DIR)
tpl = read_template(f"{FP}/template.test.bed.gz"); N_TOTAL = int(tpl.len.sum())
SEQ, LOW = read_genome(f"{BASE}/genome/genome.fa.gz", set(table.chrom) | set(tpl.chrom))
wl_train = pd.read_csv(f"{FP}/whitelist.train.bed.gz", sep="\t", header=None, usecols=[0, 1, 2], names=["chrom", "start", "end"], dtype={"chrom": str})

## 1. Признаки для всех строк таблицы

61 признак храним в `float32`, логотип и пары букв — в `uint8` (это 0/1), чтобы всё поместилось в память. LightGBM
собирает из них обучающую выборку кусками через `lgb.Sequence`, не создавая одну огромную матрицу.

In [ ]:
rows = table if len(table) <= MAX_TRAIN_ROWS else pd.concat([table[table.y == 1], table[table.y == 0].sample(MAX_TRAIN_ROWS - int(table.y.sum()), random_state=0)]).sort_index()
rows = rows.reset_index(drop=True)
t0 = time.time()
XB = np.zeros((len(rows), len(BASE_FEATURES)), np.float32)
XE = np.zeros((len(rows), len(EXTRA)), np.uint8)
def extra(seq, pos, strand):
    return np.hstack([F.logo_onehot(seq, pos, strand), F.dinuc_onehot(seq, pos, strand)])
for c, d in tqdm(rows.groupby("chrom", sort=False), desc="контиги", total=rows.chrom.nunique(), mininterval=30):
    r = d.index.values
    XB[r] = F.feature_matrix(SEQ[c], LOW[c], d.pos.values, d.strand.values, PARAMS, BASE_FEATURES)
    for st in ("+", "-"):
        m = d.strand.values == st
        if m.any(): XE[r[m]] = extra(SEQ[c], d.pos.values[m], st)
print(f"{len(rows):,} строк за {(time.time() - t0) / 60:.1f} мин; память {(XB.nbytes + XE.nbytes) / 1e9:.1f} ГБ")

class Rows(lgb.Sequence):
    # «ленивая» матрица: строки idx из XB (float) и XE (0/1), склеенные по требованию
    def __init__(self, idx, batch_size=200_000): self.idx, self.batch_size = idx, batch_size
    def __len__(self): return len(self.idx)
    def __getitem__(self, k):
        j = self.idx[k]
        if np.ndim(j) == 0: return np.concatenate([XB[j], XE[j]]).astype(np.float64)   # lgb.Sequence требует float64
        return np.hstack([XB[j], XE[j]]).astype(np.float64)

LGB = dict(max_depth=6, num_leaves=64, learning_rate=0.05, min_data_in_leaf=200, bagging_fraction=0.8, bagging_freq=1,
           feature_fraction=0.8, verbose=-1, num_threads=os.cpu_count())
N_TREES = 800

def train_pair(idx, seed):
    # (шанс старта, сила старта среди стартов) на строках idx
    y, tot = rows.y.values[idx], rows.tot.values[idx]
    cls = lgb.train({**LGB, "objective": "binary", "seed": seed}, lgb.Dataset(Rows(idx), label=y, feature_name=ALL, free_raw_data=True), N_TREES)
    pidx = idx[y == 1]
    reg = lgb.train({**LGB, "objective": "regression", "seed": seed}, lgb.Dataset(Rows(pidx), label=np.log1p(rows.tot.values[pidx]), feature_name=ALL, free_raw_data=True), N_TREES)
    return cls, reg

## 2. Честная оценка: одна модель на `fit`, плотная валидация, выбор квантования

In [ ]:
t0 = time.time()
fit_idx = np.flatnonzero(rows.split.values == "fit")
MODEL_FIT = [train_pair(fit_idx, seed=0)]
print(f"модель на fit обучена за {(time.time() - t0) / 60:.1f} мин")

def predict_positions(models, seq, low, pos, strand, chunk=1_000_000):
    base = F.feature_matrix(seq, low, pos, np.full(len(pos), strand), PARAMS, BASE_FEATURES)
    out = np.zeros(len(pos), np.float32)
    for a in range(0, len(pos), chunk):
        b = min(a + chunk, len(pos))
        Xc = np.hstack([base[a:b], extra(seq, pos[a:b], strand).astype(np.float32)])
        out[a:b] = np.mean([np.log(np.clip(c.predict(Xc), 1e-12, 1)) + r.predict(Xc) for c, r in models], axis=0)
    return out

dense_y, dense_s, n_dense = [], [], 0
lab_all = table[table.y == 1]
for c in meta["VAL_CONTIGS"]:
    L = len(SEQ[c]); inwl = np.zeros(L, bool)
    for s, e in zip(*wl_train[wl_train.chrom == c][["start", "end"]].values.T): inwl[s:e] = True
    inwl[L // 2:] = False
    dpos = np.flatnonzero(inwl)
    if len(dpos) == 0: continue
    lab = lab_all[lab_all.chrom == c]
    tot = dict(zip(lab.pos.values * 2 + (lab.strand.values == "-"), lab.tot.values))
    for st in ("+", "-"):
        dense_s.append(predict_positions(MODEL_FIT, SEQ[c], LOW[c], dpos, st))
        dense_y.append(np.array([tot.get(k, 0.0) for k in (dpos * 2 + (st == "-")).tolist()], np.float32))
    n_dense += 2 * len(dpos)
    if n_dense >= 12_000_000: break
D_S, D_Y = np.concatenate(dense_s), np.concatenate(dense_y)
ALPHAS = (0.3, 0.4, 0.5, 0.55, 0.6, 0.65, 0.7)
grid = pd.DataFrame([{"alpha": a, **S.official_score(S.rank_power(D_S, a), D_Y)} for a in ALPHAS]).set_index("alpha")
grid["AUPRC / лучший"] = grid.AUPRC / grid.AUPRC.max()
ALPHA = float(grid[grid["AUPRC / лучший"] >= 0.995].Spearman.idxmax())
display(grid.style.format("{:.4f}"))
print(f"плотная валидация: {len(D_Y):,} позиций, стартов {(D_Y > 0).sum():,}; alpha = {ALPHA}: "
      f"AUPRC {grid.loc[ALPHA, 'AUPRC']:.4f}, Spearman {grid.loc[ALPHA, 'Spearman']:.4f}  ({(time.time() - t0) / 60:.1f} мин)")
del MODEL_FIT, dense_s

COMMENT_DENSE

## 3. Финальные модели: все train-контиги, несколько сидов

In [ ]:
t0 = time.time()
all_idx = np.arange(len(rows))
MODELS = [train_pair(all_idx, seed=s) for s in range(N_SEEDS)]
for k, (c, r) in enumerate(MODELS):
    c.save_model(f"{OUT_DIR}/{SPECIES}.step6_cls_seed{k}.txt"); r.save_model(f"{OUT_DIR}/{SPECIES}.step6_reg_seed{k}.txt")
print(f"{N_SEEDS} пары моделей на {len(all_idx):,} строках за {(time.time() - t0) / 60:.1f} мин")
del XB, XE

## 4. Тест и сабмит

In [ ]:
SCORE = np.zeros(N_TOTAL, np.float32); NMASK = np.zeros(N_TOTAL, bool)
t0 = time.time()
groups = tpl.groupby(["chrom", "strand"], sort=False)
for (c, st), g in tqdm(groups, desc="контиг × цепь", total=groups.ngroups, mininterval=60):
    pos = np.concatenate([np.arange(s, e) for s, e in zip(g.start.values, g.end.values)])
    dst = np.concatenate([np.arange(o, o + n) for o, n in zip(g.off.values, g.len.values)])
    SCORE[dst] = predict_positions(MODELS, SEQ[c], LOW[c], pos, st)
    NMASK[dst] = SEQ[c][pos] == 4
SCORE[NMASK] = -np.inf
np.save(f"{OUT_DIR}/{SPECIES}.step6_score.npy", SCORE)
SUB = S.rank_power(SCORE, ALPHA, NMASK)
SUB_PATH = f"{OUT_DIR}/{SPECIES}.simple_model_final.predictions.txt.gz"
S.write_submission(SUB, SUB_PATH)
print(f"тест: {N_TOTAL:,} позиций за {(time.time() - t0) / 60:.1f} мин -> {SUB_PATH} ({os.path.getsize(SUB_PATH) / 1e6:.0f} МБ)")
json.dump({"alpha": ALPHA, "n_seeds": N_SEEDS, "train_rows": len(rows), "features": ALL,
           "dense": dict(grid.loc[ALPHA, ["AUPRC", "Spearman"]])}, open(f"{OUT_DIR}/{SPECIES}.step6_summary.json", "w"), default=float)

## 5. Проверка пайплайна по выданным файлам и официальным скорером

Официальные инструменты организаторов (https://github.com/autosome-ru/EPIC_scoring):
* `validate N файл` — проверяет, что в файле ровно N строк и каждая в формате `0.xxxxx`;
* `scorer.py` — считает метрики; с `--mask` и `--mask-mode both` — отдельно по двум половинам теста.

Что проверяем:
1. **Пример сабмита** `template.test.txt.gz` из архива и **наш файл** проходят `validate`.
2. **Train-шаблон.** Пишем файл по `template.train.bed.gz` (311 млн позиций у *Nematostella*) из простого счёта мотива
   Initiator (`inr_pwm`, считается за минуты) и сравниваем с ответами `csRNA-train.txt.gz`. Если бы порядок строк
   или цепей где-то был перепутан, AUPRC упал бы до случайного уровня. Заодно сверяем официальный скорер с нашим кодом
   на другом шаблоне. Это проверка пайплайна, не оценка качества.
3. **Тест** (если есть ответы): официальный скорер на финальном сабмите, по половинам.

In [ ]:
TOOLS = "/tmp/EPIC_scoring"
if not os.path.isdir(TOOLS):
    subprocess.run(["git", "clone", "-q", "https://github.com/autosome-ru/EPIC_scoring.git", TOOLS], check=True)
def validate(n, path):
    return subprocess.run(["bash", f"{TOOLS}/validate", str(n), path], capture_output=True, text=True).stdout.strip()
def official(gt, pred, mask=None):
    cmd = [sys.executable, f"{TOOLS}/scorer.py", "--ground-truth", gt, "--prediction", pred]
    if mask: cmd += ["--mask", mask, "--mask-mode", "both"]
    out = subprocess.run(cmd, capture_output=True, text=True)
    return json.loads(out.stdout.strip().splitlines()[-1])

checks = {}
if os.path.exists(f"{FP}/template.test.txt.gz"):
    checks["validate: пример template.test.txt.gz"] = validate(N_TOTAL, f"{FP}/template.test.txt.gz")
checks["validate: наш финальный сабмит"] = validate(N_TOTAL, SUB_PATH)

# train-шаблон: файл из счёта Initiator
TRAIN_GT = f"{FP}/csRNA-train.txt.gz"
if os.path.exists(f"{FP}/template.train.bed.gz") and os.path.exists(TRAIN_GT):
    t0 = time.time()
    ttr = read_template(f"{FP}/template.train.bed.gz"); N_TRAIN = int(ttr.len.sum())
    inr = np.zeros(N_TRAIN, np.float32)
    for (c, st), g in ttr.groupby(["chrom", "strand"], sort=False):
        o, _ = F.oriented(SEQ[c], LOW[c], st); trk = F.pwm_track(o, PARAMS["pwm"]); L = len(o)
        for s, e, off in zip(g.start.values, g.end.values, g.off.values):
            p = np.arange(s, e); inr[off:off + e - s] = trk[F.to_index(p, L, st)]
    train_sub = f"{OUT_DIR}/{SPECIES}.train_template_inr.txt.gz"
    S.write_submission(S.rank_power(inr, 0.0), train_sub)
    checks["validate: наш файл для train-шаблона"] = validate(N_TRAIN, train_sub)
    gt_train = pd.read_csv(TRAIN_GT, header=None, dtype=np.float32)[0].values
    ours = S.official_score(S.rank_power(inr, 0.0), gt_train)
    off_tr = official(TRAIN_GT, train_sub)
    checks["train-шаблон, счёт Initiator: AUPRC наш / официальный / случайный"] = f"{ours['AUPRC']:.5f} / {off_tr['prauc']:.5f} / {(gt_train > 0).mean():.5f}"
    checks["train-шаблон, счёт Initiator: Spearman наш / официальный"] = f"{ours['Spearman']:.4f} / {off_tr['spearman']:.4f}"
    print(f"train-шаблон: {N_TRAIN:,} позиций, {(time.time() - t0) / 60:.1f} мин")
    del inr, gt_train

if os.path.exists(f"{BASE}/ground_truth/csRNA-test.txt.gz"):
    GT_PATH, M1 = f"{BASE}/ground_truth/csRNA-test.txt.gz", f"{BASE}/scoring/mask1-test.txt.gz"
    off = official(GT_PATH, SUB_PATH, M1)
    gt = pd.read_csv(GT_PATH, header=None, dtype=np.float32)[0].values
    m1 = pd.read_csv(M1, header=None, dtype=np.int8)[0].values.astype(bool)
    ours = {h: S.official_score(SUB, gt, m) for h, m in (("half1", m1), ("half2", ~m1))}
    for h, k in (("half1", "mask"), ("half2", "inverted_mask")):
        checks[f"тест {h}: AUPRC наш / официальный"] = f"{ours[h]['AUPRC']:.5f} / {off[k]['prauc']:.5f}"
        checks[f"тест {h}: Spearman наш / официальный"] = f"{ours[h]['Spearman']:.4f} / {off[k]['spearman']:.4f}"
    TEST_RESULT = {"AUPRC": np.mean([off["mask"]["prauc"], off["inverted_mask"]["prauc"]]),
                   "Spearman": np.mean([off["mask"]["spearman"], off["inverted_mask"]["spearman"]])}
pd.Series(checks, name="результат").to_frame()

In [ ]:
if "TEST_RESULT" in globals():
    REF = {"динуклеотиды (бейзлайн организаторов)": (0.00116, 0.0361), "шаг 4: 61 признак, глубина 3": (0.0201, 0.1955),
           "шаг 5: + логотип, глубина 6": (0.0292, 0.2738), "InitiationNet big 96k (лучшая CNN)": (0.13165, 0.391)}
    cmp = pd.DataFrame({**{k: {"AUPRC": a, "Spearman": s} for k, (a, s) in REF.items()}, "шаг 6: финальная": TEST_RESULT}).T
    display(cmp.style.format("{:.4f}"))

COMMENT_SUMMARY